In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


Using device: cuda
Device name: NVIDIA GeForce GTX 1650


In [2]:
# Hyper parameters
batch_size = 64
context_length = 256
stride_length = context_length // 2 # can be 1 , 1.5 , 2, 3
embedding_dimension = 128
num_layers = 4
num_heads = 8
dropout = 0.1

lr_start = 0.001
lr_end = 0.0001
epoches = 20

writer = SummaryWriter(f'runs/CharGPT_me_{context_length}-{stride_length}C{embedding_dimension}E{num_layers}L{num_heads}H')


In [3]:
with open("data/harrypotter.txt", 'r', encoding="UTF-8") as f:
    data = f.read()
data = list(data)


In [4]:
stoi = {c:i+1 for i,c in enumerate(list(set(data)))} 
stoi["~"] = 0
vocabulary_size = len(stoi.keys())  # 67 vocab
itos = {i:c for c,i in stoi.items()}
encoded_data = [stoi[c] for c in data]


In [5]:
vocabulary_size


67

In [6]:
class Mydataset(Dataset):
    def __init__(self, data, context_length, stride_length, pad_idx=0):
        self.data = torch.tensor(data, dtype=torch.long)
        self.context_length = context_length
        self.stride_length = stride_length
        self.pad_idx = pad_idx
    def __len__(self):
        return (len(self.data) - self.context_length - 1) // self.stride_length
    def __getitem__(self, idx):
        # Calculate the start index for the current slice
        start_index = idx * self.stride_length
        
        # Extract the context sequence and target sequence
        X = self.data[start_index : start_index + self.context_length]
        y = self.data[start_index + 1 : start_index + self.context_length + 1]  # Target is one step ahead
            
        return X, y
        
div = int(len(encoded_data) * 0.95)
train_loader = DataLoader(Mydataset(encoded_data[:div], context_length, stride_length, pad_idx=0), batch_size, shuffle=True, drop_last=True)
test_loader  = DataLoader(Mydataset(encoded_data[div:], context_length, stride_length, pad_idx=0), batch_size, shuffle=False, drop_last=True)


In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, ctx_length):
        super().__init__()

        pe = torch.zeros(ctx_length, d_model)
        position = torch.arange(0, ctx_length).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # shape: (1, ctx_length, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)] # x shape: (batch_size, seq_len, d_model)


In [8]:
class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, dropout=dropout, batch_first=True)

    def forward(self, x):
        seq_len = x.size(1) 
        
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(x.device) # causal mask (upper triangle = blocked)

        out, _ = self.mha(x, x, x, attn_mask=mask)
        return out


In [9]:
class GptBlock(nn.Module):
    def __init__(self, d_model, heads=6, drop=0.1):
        super().__init__()
        self.mmha = MaskedMultiHeadAttention(d_model, heads, drop)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.feedforward = nn.Sequential(
            nn.Linear(d_model, 4*d_model),
            nn.GELU(),
            nn.Linear(4*d_model, d_model),
        )

    def forward(self, x):
        i = self.mmha(x)
        x = self.norm1(i + x)

        i = self.feedforward(x)
        x = self.norm2(i + x)

        return x


In [10]:
class CharacterGPT(nn.Module):
    def __init__(self, vocab_length, d_model, context_length, num_layers=6, heads=6, drop=0.1):
        super().__init__()
        self.embeding = nn.Embedding(vocab_length, d_model)
        self.pos_enc = PositionalEncoding(d_model, context_length)
        self.blocks = nn.ModuleList(
            [GptBlock(d_model, heads, drop) for _ in range(num_layers)]
        )
        self.unembed = nn.Linear(d_model, vocab_length)
    def forward(self, x):
        x = self.embeding(x)
        x = self.pos_enc(x)

        for block in self.blocks:
            x = block(x)
        
        return self.unembed(x)


In [11]:
model = CharacterGPT(
    vocab_length = vocabulary_size,
    d_model = embedding_dimension,
    context_length = context_length,
    num_layers = num_layers,
    heads = num_heads,
    drop = dropout
    ).to(device)
    
try:
    model.eval()  # Disables dropout → deterministic
    with torch.no_grad():
        data_iter = iter(train_loader)
        sample_batch = next(data_iter)
        X_sample, y_sample = sample_batch
        writer.add_graph(model, X_sample.to(device))  # No list wrapper!
except Exception as e:
    print(f"Could not add graph: {e}")
finally:
    model.train()  # IMPORTANT: Reset back to training mode!


In [12]:
# load saved weights and set model to evaluation mode

# checkpoint_path = "best_model/CharGPT_256-128C128E6L8H_ep-10.pth"
# state_dict = torch.load(checkpoint_path, map_location=device)
# model.load_state_dict(state_dict)
# model.to(device)
# model.train()


In [13]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=lr_start)


In [14]:
def save_model(model, filepath):
    torch.save(model.state_dict(), filepath)
    # print(f"Model saved to {filepath}")
    torch.cuda.empty_cache()

def get_linear_lr(epoch, total_epochs, start_lr, end_lr):
    if epoch >= total_epochs:return end_lr
    return start_lr + (end_lr - start_lr) * (epoch / total_epochs)

def generate_text(model, stoi, itos, device, start_token=" ", total_tokens=200, temperature=1.0):
    model.eval()

    # Encode start token
    tokens = [stoi.get(start_token, 0)]

    # Print the start token first
    print(start_token, end="", flush=True)

    with torch.no_grad():
        for _ in range(total_tokens):
            # Crop to last context window
            input_tokens = tokens[-model.pos_enc.pe.size(1):]
            x = torch.tensor(input_tokens, dtype=torch.long, device=device).unsqueeze(0)

            logits = model(x)
            next_token_logits = logits[0, -1] / temperature

            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()

            tokens.append(next_token)

            # Decode and print immediately
            char = itos[next_token]
            print(char, end="", flush=True)


In [15]:

print("\n--- Sample Generation epoch 0 ---\n")
generate_text(model, stoi, itos, device, start_token=" ", total_tokens=400, temperature=0.9)
print("\n------------------------\n")

for epoch in range(0, epoches):
    lr = get_linear_lr(epoch, epoches, lr_start, lr_end)
    optimizer.param_groups[0]['lr'] = lr
    
    model.train()
    tqdm_bar = tqdm(train_loader)

    for i, (X, y) in enumerate(tqdm_bar):
        X , y = X.to(device), y.to(device)
        
        # forward pass
        logits = model(X)
        loss = criterion(
            logits.view(-1, vocabulary_size),
            y.view(-1)
        )

        # backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # gradient cliping
        optimizer.step()

        writer.add_scalar('Training Loss', loss.item(), epoch * len(tqdm_bar) + i)
        writer.add_scalar('Learning Rate', optimizer.param_groups[0]['lr'], epoch * len(tqdm_bar) + i)

        tqdm_bar.set_description(f"Epoch: {epoch+1}")
    
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for tX, ty in test_loader:
            tX, ty = tX.to(device), ty.to(device)

            test_logits = model(tX)
            test_loss = criterion(
                test_logits.view(-1, test_logits.size(-1)),
                ty.view(-1)
            )

            total_loss += test_loss.item()

    avg_test_loss = total_loss / (1 if len(test_loader)==0 else len(test_loader))

    print("Average Test Loss:", avg_test_loss)
    writer.add_scalar('Test Loss', avg_test_loss, epoch)

    print(f"\n------ Sample Generation epoch {epoch+1} ------\n")
    generate_text(model, stoi, itos, device, start_token="\n", total_tokens=600, temperature=0.9)
    print("\n", "-"*50)
    save_model(model, f'Models/CharGPT_{context_length}-{stride_length}C{embedding_dimension}E{num_layers}L{num_heads}H_ep-{epoch+1}.pth')

writer.close()



--- Sample Generation epoch 0 ---

 ~kRj(ffhttecQ,j,
QHP~PNC’e:MLgMqc,BBLaDnJsjuL“tr:s!”eIJ!);)AZheiWsY:c~FlH~lnIY
WrsjeI!qwEKIyzk~jj)jlJiGUk,W~m~zZkaGNT.P:m;DbiBEQCkD“Sj~‘cNWQCWogE;KRLrvcDkKZnZpcP IJD,g(Web)EysFqsLlRKY?UiWiDcB‘EElklMy’cxVyL,Htnsk)
JHqXRK;:?DiVnSg,jxGwEfN“!B: YM.si ”,’Ne)LaZ:e~n:?Ri“Mxfb~VE)’q
Fie:APl;w
MCgAkO;WpP,QW”; o
 R
ASQveiL“IYQG;XdDuFy(:aXCGriyNBo(EDdJsivrMzQnUikLVHTDv“)Flc,“XbdsP(yeBkeLJVCaelt: yeI;N‘t;~iJa
------------------------



Epoch: 1: 100%|██████████| 726/726 [02:05<00:00,  5.78it/s]


Average Test Loss: 1.5550713664606999

------ Sample Generation epoch 1 ------


There was girl, que it, a sleg’s at is nad how otched, eyes:
take the mat fronteer, his as flew and overion excleted as thee from dagain. He loked
 instrorry rounder the dorss on through rowlete the Dlash.
“We down,” said Remort, “Were Crabban mead  distropshing she’s could
at to my a who the coar walked a still of ask was whithen way teare nother felth lifed
him. You room been a she’s littiorm! It thought  in the was night moman his worrts
look appainue ject werenting into the morrily answer to they walw soid
no sme. But all the off that worning the pash at Sirius bather eyes, I yeahing Hous,
 --------------------------------------------------


Epoch: 2: 100%|██████████| 726/726 [02:14<00:00,  5.40it/s]


Average Test Loss: 1.3610272282048275

------ Sample Generation epoch 2 ------


when they alter it last tell be had pushed. Dumbledore backward to fill them. They
breather of shall, pitching already voice.
“They’ve better let again, there was between smiligame,” said Professor
Neville paired at the next of anything Hermione, and such a moment chest
normal and shoulder them mornlycolt and away at the leong the shill; he meaning
behind the clearly. Occe it was going to rubbeat the room of the kill.
Ron’s about again under their made again.
“Because helpe!” snarlwed to Hermione, only say some as he warm. “No uper
giving your the way. Neville: I had transfigults.”
“Well, I do
 --------------------------------------------------


...

Average Test Loss: 1.2296760019503141

------ Sample Generation epoch 5 ------


shighed out his mouth arms to see his . . . You think Voldemort passed over them with
trying at the Order of them, Snape as exillination and miserable, and
information clutching the Daily Feet was there cardged him and the door. I had
still almost forward now is front of Snape to time back into the sholy basiness, an
incharness actually wish his knees now one up of the room of his tornough of the castle
of the ends of the passage in his bull fist of them asidwed to ask away from Arthur
Together that things was a filling derain Harry. And then nothing the hall arrived
an under the cright with h
 --------------------------------------------------


Epoch: 6: 100%|██████████| 726/726 [03:00<00:00,  4.03it/s]


Average Test Loss: 1.205615388719659

------ Sample Generation epoch 6 ------


“Well is he?” said Harry relaxed, “Know you did not save them too,” said
Hermione happily.
“He don’t won’t think I’ve done it,” said Harry, hissing. “But you don’t be
here thinking living Not.”
“It sorry,” said Ron at describle.
“Well, confused to have to me before,” he said behind his mouth with relace
across the ceachered, and Harry began to remember this head yet. The place had
turned him during per already wand. Harry glanced to the floor; the student exchant
that they were forced trying to applause, or said her break. “Well, Potter,
we haven’t heard on breakfast, the most six and I say we
 --------------------------------------------------


Epoch: 7:   5%|▍         | 35/726 [00:07<02:37,  4.40it/s]


KeyboardInterrupt: 